# Image and Video Analysis. Project 2: fish detection using CamShift
## Authors: Martín Ignacio Rizzo and Antoni Ginard Sureda

# Introduction
Tracking objects in video sequences is a common but challenging task in computer vision. Nowadays, with methods like Deep Learning (DL) this tasks has become easier in comparison with previous years. However, these DL models are resource-intensive
and may not always be the best solutions for real-time applications. Because of these, algorithms like MeanShift or CamShift might be a better option for real-time applications.

The objective of this project is to implement an object tracking system in a real-world problem using an algorithm seen in class.

## Context
The scenario chosen in this project is the tracking of fish in a video sequence. Given that there was no chance of recording fish in the sea, a video of fish in a tank was used to simulate the real-world problem. This tank has several fished swimming
and objects emulating the environment of the sea. Being able to track fish in the sea would be a real-problem for aquaculture or fish research. ## TODO: Add more context

# Implementation
The tracking algorithm chosen in this project is CamShift. CamShift is an extension of MeanShift that provides better tracking performance.


In [ ]:
VIDEO_PATH = "peixos2.mp4"

SCENE_WIDTH_M = 1.2
SCENE_HEIGHT_M = 0.7

In [ ]:
def center_of_rect(rect):
    """Returns the center of a rectangle (x, y, w, h)."""
    x, y, w, h = rect
    cx = x + w / 2.0
    cy = y + h / 2.0
    return (cx, cy)

In [ ]:
import cv2

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError(f"Can't open video {VIDEO_PATH}")

Read the first frame and allow the user to select the ROIs

In [ ]:
ret, first_frame = cap.read()
if ret:
    raise RuntimeError("Can't read first frame of the video")

frame_h, frame_w = first_frame.shape[:2]

meters_per_pixel_x = SCENE_WIDTH_M / frame_w
meters_per_pixel_y = SCENE_HEIGHT_M / frame_h

fps = cap.get(cv2.CAP_PROP_FPS)
if fps == 0:
    fps = 30.0

# The video is in high resolution, which would make it hard to display. Thus,
# limit its size to 1280x720 and the scale the selected ROIs accordingly
max_display_w = 1280
max_display_h = 720
roi_scale = min(max_display_w / frame_w, max_display_h / frame_h, 1.0)
first_frame_resized = cv2.resize(first_frame, None, fx=roi_scale, fy=roi_scale)

rois = cv2.selectROIs("Select fish", first_frame_resized, fromCenter=False, showCrosshair=True)
cv2.destroyWindow("Select fish")

rois = list(rois)

if len(rois) == 0:
    raise RuntimeError("No ROIs selected")

if roi_scale != 1.0:
    rois = [
        (
            int(round(x / roi_scale)),
            int(round(y / roi_scale)),
            int(round(w / roi_scale)),
            int(round(h / roi_scale)),
        )
        for (x, y, w, h) in rois
    ]


Each fish has its own tracking structure, which includes:
- The track window (x, y, w, h)
- The histogram of the fish
- The previous center of the fish
- The velocity of the fish

The following code initializes all the needed structures.

In [ ]:
fish_data = []

hsv_first = cv2.cvtColor(first_frame, cv2.COLOR_BGR2HSV)

for roi in rois:
    x, y, w, h = roi

    # ROI in HSV
    roi_hsv = hsv_first[y:y + h, x:x + w]

    # Ignore darker and brighter pixels
    mask = cv2.inRange(roi_hsv, (0, 30, 30), (180, 255, 255))

    # ROI histogram from the H channel
    roi_hist = cv2.calcHist([roi_hsv], [0], mask, [180], [0, 180])
    cv2.normalize(roi_hist, roi_hist, 0, 255, cv2.NORM_MINMAX)

    # Initial tracking window for CamShift
    track_window = (x, y, w, h)

    fish_data.append({
        "track_window": track_window,
        "roi_hist": roi_hist,
        "prev_center": center_of_rect(track_window),
        "velocity_m_s": (0.0, 0.0)  # vx, vy in m/s
    })


Now the actual tracking pipeline can start. It consists in:
1. Reading the video frame by frame
2. Substracting the background of the frame using the Mixture of Gaussians (MOG) version 2 background substractor
3. Convert the frame to HSV
4. For each fish, calculate the backprojection of the histogram of the fish
5. Apply CamShift to track the fish
6. Draw the bounding box of the fish
7. Calculate the velocity of the fish
8. Draw the velocity vector of the fisho
9. Show the frame with each bounding box and information

The following code implements three auxiliar functions that are used in the main tracking pipeline:

In [ ]:
import math


def extract_background(substractor, frame):
    fg_mask = substractor.apply(frame)
    fg_mask = cv2.threshold(fg_mask, 180, 255, cv2.THRESH_BINARY)[1]

    masked_frame = cv2.bitwise_and(frame, frame, mask=fg_mask)
    return masked_frame

def compute_bounding_box(pts):
    xs = pts[:, 0]
    ys = pts[:, 1]
    x_min, x_max = xs.min(), xs.max()
    y_min, y_max = ys.min(), ys.max()
    w_box = x_max - x_min
    h_box = y_max - y_min

    return (x_min, y_min, w_box, h_box)

def draw_arrow(image, p_start, p_end, color=(0, 0, 255), thickness=2):
    """Draws an arrow from p_start to p_end."""
    p_start = (int(p_start[0]), int(p_start[1]))
    p_end = (int(p_end[0]), int(p_end[1]))
    cv2.arrowedLine(image, p_start, p_end, color, thickness, tipLength=0.3)

def compute_speed(prev_center, curr_center):
    dx_pixels = curr_center[0] - prev_center[0]
    dy_pixels = curr_center[1] - prev_center[1]

    dx_pixels_per_s = dx_pixels * fps
    dy_pixels_per_s = dy_pixels * fps

    vx_m_s = dx_pixels_per_s * meters_per_pixel_x
    vy_m_s = dy_pixels_per_s * meters_per_pixel_y

    speed_m_s = math.sqrt(vx_m_s ** 2 + vy_m_s ** 2)

    return vx_m_s, vy_m_s, speed_m_s

With all the requirements satisfied, the main tracking loop can start.

In [ ]:
import numpy as np

# Terminaiton criteria for CamShift
term_crit = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 1)
back_sub = cv2.createBackgroundSubtractorMOG2()

while True:
    ret, frame = cap.read()

    if not ret:
        break

    masked_frame = extract_background(back_sub, frame)

    hsv = cv2.cvtColor(masked_frame, cv2.COLOR_BGR2HSV)

    for i, fish in enumerate(fish_data):
        track_window = fish["track_window"]
        roi_hist = fish["track_window"]
        prev_center = fish["prev_center"]

        # Compute backprojection
        back_proj = cv2.calcBackProject([hsv], [0], roi_hist, [0, 180], 1)

        # Apply CamShift
        ret_camshift, new_window = cv2.CamShift(back_proj, track_window, term_crit)
        fish["track_window"] = new_window

        pts = cv2.boxPoints(ret_camshift)
        pts = pts.astype(np.int32)

        x, y, w, h = compute_bounding_box(pts)

        curr_center = center_of_rect((x, y, w, h))

        cv2.rectangle(frame, (x, y), (x + w,  y + h), (0, 255, 0), 2)

        vx_m_s, vy_m_s, speed_m_s = compute_speed(prev_center, curr_center)

        fish["velocity_m_s"] = (vx_m_s, vy_m_s)
        fish["prev_center"] = curr_center

        scale = 0.5
        end_point = (
            curr_center[0] + vx_m_s * scale,
            curr_center[1] + vy_m_s * scale
        )
        draw_arrow(frame, curr_center, end_point, color=(0, 0, 255), thickness=2)

        cv2.circle(frame, (int(curr_center[0]), int(curr_center[1])), 3, (255, 0, 0), -1)

        text = f"{speed_m_s:.2f} m/s"
        cv2.putText(frame, text, (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1, cv2.LINE_AA)

    resized_frame = cv2.resize(frame, None, fx=0.5, fy=0.5)
    cv2.imshow("Tracking fish", resized_frame)
    key = cv2.waitKey(1) & 0xFF
    if key == 27 or key == ord('q'):
        break


In [ ]:
cap.release()
cv2.destroyAllWindows()

# Results
Results can be seen in this video: # TODO!

# Conclusions
The results are very promising. The tracking system is able to track the selected fish with good accuracy. However, there are cases where the tracking fails.
For example, when two very similar fish are close to each other, the tracking system is not able to distinguish between them, and might start tracking the
fish that wasn't selected. Also, the camouflage effect of the fish might make it hard to track the fish, as the histogram of the fish might not be able to
be different enough from the environment.
